# 🚀 Two-Agent News Sentiment Analyzer with Agent-to-Agent Delegation

This notebook implements an agent-to-agent delegation workflow for financial sentiment analysis using AutoGen's **Nested Chats**:
1. Python fetches raw articles and queries the FAISS vector database for calibration examples.
2. The user initiates a chat with the **Senior Sentiment Analyst (CIO) Agent**.
3. The CIO Agent automatically triggers a nested chat, delegating the scoring task to the **Sentiment Scorer Agent**.
4. The Scorer Agent analyzes the articles, calculates scores, and returns them to the CIO Agent.
5. The CIO Agent aggregates the scores, averages the sentiment, and returns the final JSON report back to the user.

In [ ]:
import sys
import os
from dotenv import load_dotenv

# Ensure the current directory is in the python path for importing modules
notebook_dir = os.getcwd()
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

sentiment_dir = os.path.dirname(notebook_dir)
if sentiment_dir not in sys.path:
    sys.path.insert(0, sentiment_dir)

project_root = os.path.dirname(sentiment_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Load environment variables from .env.local
load_dotenv("../.env.local")

In [ ]:
import json
import datetime
import pandas as pd
import autogen
from finrobot.agents.workflow import FinRobot
from autogen import UserProxyAgent
from sentiment.functions.aggregator.aggregator import fetch_aggregate_all_news
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

# Import refactored utility functions
from sentiment.functions.utils.read_and_clean import read_file_content, extract_and_clean_response
from sentiment.functions.utils.build import build_vector_store
from sentiment.functions.utils.config import generate_config
from sentiment.functions.tools.prepare_articles import prepare_articles

# Read and clean environment variables
nvidia_embedding_model = os.getenv("NVIDIA_EMBEDDING_MODEL", "nvidia/nv-embed-v1").strip('"\' ')
nvidia_base_model = os.getenv("NVIDIA_BASE_MODEL", "").strip('"\' ')
nvidia_api_endpoint = os.getenv("NVIDIA_API_ENDPOINT", "https://integrate.api.nvidia.com/v1").strip('"\' ')
nvidia_api_key = os.getenv("NVIDIA_API_KEY", "").strip('"\' ')

print(f"Initializing NVIDIA Embeddings wrapper ({nvidia_embedding_model})...")
embeddings = NVIDIAEmbeddings(
    model=nvidia_embedding_model,
    nvidia_api_key=nvidia_api_key,
    base_url=nvidia_api_endpoint
)

In [ ]:
from sentiment.functions.tools.agents import create_scorer_agent, create_cio_agent

In [ ]:
hf_api_key = os.getenv("HUGGINGFACE_API_KEY", "").strip('"\' ')
hf_model_name = os.getenv("HUGGINGFACE_MODEL_NAME_FEATHERLESS", "curiousily/Llama-3-8B-Instruct-Finance-RAG").strip('"\' ')
hf_base_url = os.getenv("HUGGINGFACE_BASE_URL", "https://router.huggingface.co/v1").strip('"\' ')

print(f"HF Model Name: {hf_model_name}")
print(f"HF Base URL: {hf_base_url}")
print(f"HF API Key exists: {bool(hf_api_key)}")

config_list = generate_config(hf_model_name, hf_base_url, hf_api_key)
base_config_list = generate_config(nvidia_base_model, nvidia_api_endpoint, nvidia_api_key)

llm_config = {"config_list": config_list, "model": hf_model_name}
base_llm_config = {"config_list": base_config_list, "model": nvidia_base_model}

In [ ]:
ticker = "AAPL"
news_limit = 5  # Score top 5 articles

In [ ]:
# Build vector store
db = build_vector_store("../data/financial_sentiment.csv", embeddings, limit_rows=300)

# Instantiate the UserProxyAgent
user_proxy = UserProxyAgent(
    name="User_Proxy",
    human_input_mode="NEVER",
    is_termination_msg=lambda x: x.get("content", "") and "TERMINATE" in x.get("content", ""),
    max_consecutive_auto_reply=1,
    code_execution_config={"use_docker": False}
)

In [ ]:
# Instantiate scorer and CIO agents
scorer_agent = create_scorer_agent(
    prompt_path="../prompts/sentiment_prompt.txt",
    schema_path="../schema_json/scorer_schema.json",
    llm_config=llm_config
)

cio_agent = create_cio_agent(
    prompt_path="../prompts/cio_prompt.txt",
    schema_path="../schema_json/sentiment_schema.json",
    output_schema_path="../schema_json/cio_output_schema.json",
    scored_articles_path="../schema_json/cio_scored_articles.json",
    llm_config=base_llm_config
)

In [ ]:
# Prepare the news articles
print(f"Fetching consolidated news feed for {ticker}...")
df_news = fetch_aggregate_all_news(symbol=ticker, limit=100)

if df_news.empty:
    raise ValueError(f"No news articles found for symbol {ticker}.")
    
articles_to_analyze = prepare_articles(df_news, db, limit=news_limit)

# Configure the nested chat on the CIO Agent.
# When the User Proxy sends raw articles to the CIO, the CIO delegates them to the Scorer.
nested_chats = [
    {
        "recipient": scorer_agent,
        "message": lambda recipient, messages, sender, config: (
            "Please score the following articles according to your instructions:\n\n"
            f"{messages[-1]['content']}\n\n"
            "Respond with the list of scored articles."
        ),
        "summary_method": "last_msg",
        "max_turns": 1,
    }
]

from sentiment.functions.tools.custom_reply import custom_nested_chat_reply

cio_agent.register_nested_chats(
    nested_chats,
    trigger=user_proxy,
    reply_func_from_nested_chats=custom_nested_chat_reply
)

# Step 1 & 2: Initiate chat directly with the CIO agent.
# The user proxy passes the raw articles.
print("\n[+] Triggering Agent-to-Agent Delegation Chat...")
user_proxy.initiate_chat(
    cio_agent,
    message=json.dumps(articles_to_analyze, indent=2)
)

final_report_msg = extract_and_clean_response(user_proxy, cio_agent, is_json=True)

print("\n================ FINAL REPORT ================")
try:
    final_report = json.loads(final_report_msg)
    print(json.dumps(final_report, indent=2))
except Exception as e:
    print(f"Error parsing final report: {e}")
    print("Raw Output:")
    print(final_report_msg)

In [ ]:
print(final_report_msg)